In [1]:
import pandas as pd
import numpy as np
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
import re
import html
import emoji
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.metrics import classification_report
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
import pandas as pd

#only run once
# nltk.download('punkt')
# nltk.download('stop_words')
# nltk.download('wordnet')


In [2]:
df = pd.read_csv('judge-1377884607_tweet_product_company.csv',encoding='latin1')
df.head()

,tweet_text,emotion_in_tweet_is_directed_at,is_there_an_emotion_directed_at_a_brand_or_product
0,.@wesley83 I have a 3G iPhone. After 3 hrs twe...,iPhone,Negative emotion
1,@jessedee Know about @fludapp ? Awesome iPad/i...,iPad or iPhone App,Positive emotion
2,@swonderlin Can not wait for #iPad 2 also. The...,iPad,Positive emotion
3,@sxsw I hope this year's festival isn't as cra...,iPad or iPhone App,Negative emotion
4,@sxtxstate great stuff on Fri #SXSW: Marissa M...,Google,Positive emotion


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9093 entries, 0 to 9092
Data columns (total 3 columns):
 #   Column                                              Non-Null Count  Dtype 
---  ------                                              --------------  ----- 
 0   tweet_text                                          9092 non-null   object
 1   emotion_in_tweet_is_directed_at                     3291 non-null   object
 2   is_there_an_emotion_directed_at_a_brand_or_product  9093 non-null   object
dtypes: object(3)
memory usage: 213.2+ KB


From the Non-Null Count we can see there is 1 tweet which is a null value. That probably means that the tweet is an empty string. 

I will drop this empty string

In [4]:
df.describe()

,tweet_text,emotion_in_tweet_is_directed_at,is_there_an_emotion_directed_at_a_brand_or_product
count,9092,3291,9093
unique,9065,9,4
top,RT @mention Marissa Mayer: Google Will Connect...,iPad,No emotion toward brand or product
freq,5,946,5389


In [5]:
df['is_there_an_emotion_directed_at_a_brand_or_product'].unique()

array(['Negative emotion', 'Positive emotion',
       'No emotion toward brand or product', "I can't tell"], dtype=object)

When cleaning the text I want to do the following:
1. Remove any starting or trailing white spaces
2. Ensure every word is lower case
3. Remove any numbers
4. Remove any special characters
5. Remove any emojis (if they dont count as special characters)
6. Tokenize
7. Lemmertize
8. Remove any stopwords
9. Remove any words that repeat consecutively
10. remove any letters that repeat more than twice
11. Remove any word that is connected to the # symbol (any words connected to special symbols)
12. Remove any links they all look like this {link}
13. Remove any empty lines.
14. Remove any . symbols that repeat multiple times
15. Remove special symbols connected to numbers 5,000
16. Remove @gmail.com or @yahoo.com
17. Remove .com

In [6]:
class DropNullRows(BaseEstimator,TransformerMixin):
    #custom transformer to drop rows where the input is null because its probably a corrupt tweet or empty tweet
    def fit(self,X,y=None):
        return self
    
    def transform(self,X):
        #return the tows where the tweet is not null
        return X.dropna()



In [7]:
class TweetPreprocessor(BaseEstimator, TransformerMixin):
    def __init__(self, 
                 remove_stopwords=True, 
                 min_token_len=3, 
                 lemmatize_words=True,
                 output_format='tokens'):  # 'tokens' or 'string'
        self.lemmatizer = WordNetLemmatizer()
        self.stop_words = set(stopwords.words('english'))
        self.remove_stopwords = remove_stopwords
        self.min_token_len = min_token_len
        self.lemmatize_words = lemmatize_words
        self.output_format = output_format

    def clean_text(self, text):
        if not isinstance(text,str):
            text = str(text) if text is not None else ''
        # # 1. Unescape HTML
        # text = html.unescape(text)

        # 2. Remove emojis
        text = emoji.replace_emoji(text, replace='')

        # 3. Lowercase
        text = text.lower()

        # 4. Remove retweet tag
        text = re.sub(r'^rt\s+', '', text)

        # 5. Remove URLs
        text = re.sub(r'https?://\S+', '', text)

        # 6. Remove mentions, hashtags, and cashtags
        text = re.sub(r'[@#\$]\w+', '', text)

        # 7. Remove numbers
        text = re.sub(r'\d+', '', text)

        # 8. Reduce repeated characters (e.g. soooo → so)
        text = re.sub(r'(.)\1{2,}', r'\1', text)

        # 9. Remove non-alphabetic characters
        text = re.sub(r'[^a-z\s]', '', text)

        # 10. Normalize whitespace
        text = re.sub(r'\s+', ' ', text).strip()

        # 11. Tokenize
        tokens = word_tokenize(text)

        # 12. Filter stopwords and short tokens
        if self.remove_stopwords:
            tokens = [word for word in tokens if word not in self.stop_words]
        if self.min_token_len > 0:
            tokens = [word for word in tokens if len(word) >= self.min_token_len]

        # 13. Lemmatize
        if self.lemmatize_words:
            tokens = [self.lemmatizer.lemmatize(word) for word in tokens]

        if self.output_format == 'string':
            return ' '.join(tokens)
        return tokens

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        return X.apply(self.clean_text)


In [8]:
# # Output as strings
# preprocessor = TweetPreprocessor(output_format='string')
# df['cleaned_string'] = preprocessor.transform(df['tweet_text'])

# # Output as token lists
# preprocessor_tokens = TweetPreprocessor(output_format='tokens')
# df['cleaned_tokens'] = preprocessor_tokens.transform(df['tweet_text'])


In [9]:
df.head()

,tweet_text,emotion_in_tweet_is_directed_at,is_there_an_emotion_directed_at_a_brand_or_product
0,.@wesley83 I have a 3G iPhone. After 3 hrs twe...,iPhone,Negative emotion
1,@jessedee Know about @fludapp ? Awesome iPad/i...,iPad or iPhone App,Positive emotion
2,@swonderlin Can not wait for #iPad 2 also. The...,iPad,Positive emotion
3,@sxsw I hope this year's festival isn't as cra...,iPad or iPhone App,Negative emotion
4,@sxtxstate great stuff on Fri #SXSW: Marissa M...,Google,Positive emotion


In [10]:
df = df.dropna(subset=['tweet_text'])

In [11]:
pipeline = Pipeline([
    ('cleantweets',TweetPreprocessor(output_format='string')),
    ('vectorize',TfidfVectorizer()),
    ('model',LogisticRegression())
])

In [12]:
# Split data
X_train, X_test, y_train, y_test = train_test_split(df['tweet_text'], df['is_there_an_emotion_directed_at_a_brand_or_product'],
                                                     test_size=0.25, random_state=42)



In [13]:
# Parameter grid
param_grid = {
    'vectorize__ngram_range': [(1, 1), (1, 2)],
    'vectorize__max_df': [0.8, 1.0],
    'vectorize__min_df': [1, 2],
    'model__C': [0.1, 1, 10]  # Regularization strength
}




In [14]:
# Grid Search CV
grid = GridSearchCV(pipeline, param_grid, cv=3, scoring='f1', verbose=1)
grid.fit(X_train, y_train)

print(f"Best Params: {grid.best_params_}")

Fitting 3 folds for each of 24 candidates, totalling 72 fits


c:\Users\franc\anaconda3\envs\learn-env\lib\site-packages\sklearn\model_selection\_validation.py:821: UserWarning: Scoring failed. The score on this train-test partition for these parameters will be set to nan. Details: 
Traceback (most recent call last):
  File "c:\Users\franc\anaconda3\envs\learn-env\lib\site-packages\sklearn\model_selection\_validation.py", line 810, in _score
    scores = scorer(estimator, X_test, y_test)
  File "c:\Users\franc\anaconda3\envs\learn-env\lib\site-packages\sklearn\metrics\_scorer.py", line 266, in __call__
    return self._score(partial(_cached_call, None), estimator, X, y_true, **_kwargs)
  File "c:\Users\franc\anaconda3\envs\learn-env\lib\site-packages\sklearn\metrics\_scorer.py", line 355, in _score
    return self._sign * self._score_func(y_true, y_pred, **scoring_kwargs)
  File "c:\Users\franc\anaconda3\envs\learn-env\lib\site-packages\sklearn\utils\_param_validation.py", line 214, in wrapper
    return func(*args, **kwargs)
  File "c:\Users\fran

Best Params: {'model__C': 0.1, 'vectorize__max_df': 0.8, 'vectorize__min_df': 1, 'vectorize__ngram_range': (1, 1)}


In [15]:
df.head()

,tweet_text,emotion_in_tweet_is_directed_at,is_there_an_emotion_directed_at_a_brand_or_product
0,.@wesley83 I have a 3G iPhone. After 3 hrs twe...,iPhone,Negative emotion
1,@jessedee Know about @fludapp ? Awesome iPad/i...,iPad or iPhone App,Positive emotion
2,@swonderlin Can not wait for #iPad 2 also. The...,iPad,Positive emotion
3,@sxsw I hope this year's festival isn't as cra...,iPad or iPhone App,Negative emotion
4,@sxtxstate great stuff on Fri #SXSW: Marissa M...,Google,Positive emotion


In [16]:
# Predict on test set
y_pred = grid.predict(X_test)
y_proba = grid.predict_proba(X_test)[:, 1]  # Probability of class 1 (positive)

# Classification report
print(classification_report(y_test, y_pred))




                                    precision    recall  f1-score   support

                      I can't tell       0.00      0.00      0.00        39
                  Negative emotion       0.00      0.00      0.00       151
No emotion toward brand or product       0.61      0.97      0.75      1320
                  Positive emotion       0.71      0.17      0.27       763

                          accuracy                           0.62      2273
                         macro avg       0.33      0.29      0.26      2273
                      weighted avg       0.59      0.62      0.53      2273



c:\Users\franc\anaconda3\envs\learn-env\lib\site-packages\sklearn\metrics\_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
c:\Users\franc\anaconda3\envs\learn-env\lib\site-packages\sklearn\metrics\_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
c:\Users\franc\anaconda3\envs\learn-env\lib\site-packages\sklearn\metrics\_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


In [17]:
# Save predictions to CSV
results = pd.DataFrame({
    'tweet': X_test,
    'actual': y_test.values,
    'predicted': y_pred,
    'probability_positive': y_proba
})
results.to_csv('tweet_predictions.csv', index=False)
print("Predictions saved to tweet_predictions.csv")

Predictions saved to tweet_predictions.csv


In [18]:
pipeline2 = Pipeline([
    ('cleantweets',TweetPreprocessor(output_format='string')),
    ('vectorize',TfidfVectorizer()),
    ('model',RandomForestClassifier())
])

In [19]:
param_grid = {
    'vectorize__max_df': [0.75, 1.0],                  # Ignore very frequent words
    'vectorize__min_df': [1, 2],                       # Include words that appear at least n times
    'vectorize__ngram_range': [(1, 1), (1, 2)],        # Unigrams vs unigrams + bigrams
    # 'model__n_estimators': [100, 200],                 # Number of trees
    # 'model__max_depth': [None, 10, 20],                # Depth of trees
    # 'model__min_samples_split': [2, 5],                # Minimum samples to split an internal node
    # 'model__min_samples_leaf': [1, 2]                  # Minimum samples in a leaf
}


In [20]:


grid2 = GridSearchCV(
    estimator=pipeline2,
    param_grid=param_grid,
    scoring='f1',     # or 'f1_macro' if it's multi-class
    cv=3,
    verbose=1,
    n_jobs=-1
)


In [21]:
# Grid Search CV
grid2 = GridSearchCV(pipeline2, param_grid, cv=3, scoring='f1', verbose=1)
grid2.fit(X_train, y_train)

print(f"Best Params: {grid2.best_params_}")

Fitting 3 folds for each of 8 candidates, totalling 24 fits


c:\Users\franc\anaconda3\envs\learn-env\lib\site-packages\sklearn\model_selection\_validation.py:821: UserWarning: Scoring failed. The score on this train-test partition for these parameters will be set to nan. Details: 
Traceback (most recent call last):
  File "c:\Users\franc\anaconda3\envs\learn-env\lib\site-packages\sklearn\model_selection\_validation.py", line 810, in _score
    scores = scorer(estimator, X_test, y_test)
  File "c:\Users\franc\anaconda3\envs\learn-env\lib\site-packages\sklearn\metrics\_scorer.py", line 266, in __call__
    return self._score(partial(_cached_call, None), estimator, X, y_true, **_kwargs)
  File "c:\Users\franc\anaconda3\envs\learn-env\lib\site-packages\sklearn\metrics\_scorer.py", line 355, in _score
    return self._sign * self._score_func(y_true, y_pred, **scoring_kwargs)
  File "c:\Users\franc\anaconda3\envs\learn-env\lib\site-packages\sklearn\utils\_param_validation.py", line 214, in wrapper
    return func(*args, **kwargs)
  File "c:\Users\fran

Best Params: {'vectorize__max_df': 0.75, 'vectorize__min_df': 1, 'vectorize__ngram_range': (1, 1)}


In [22]:
print("Best Params:", grid2.best_params_)
print("Best F1 Score:", grid2.best_score_)

y_pred2 = grid2.predict(X_test)
y_proba2 = grid2.predict_proba(X_test)[:, 1]  # Probability of class 1 (positive)

# Classification report
print(classification_report(y_test, y_pred2))

Best Params: {'vectorize__max_df': 0.75, 'vectorize__min_df': 1, 'vectorize__ngram_range': (1, 1)}
Best F1 Score: nan
                                    precision    recall  f1-score   support

                      I can't tell       0.00      0.00      0.00        39
                  Negative emotion       0.68      0.19      0.29       151
No emotion toward brand or product       0.68      0.86      0.76      1320
                  Positive emotion       0.62      0.45      0.53       763

                          accuracy                           0.66      2273
                         macro avg       0.50      0.37      0.39      2273
                      weighted avg       0.65      0.66      0.64      2273



In [ ]:
grid2.

nan